# ASCII art POC

`real image -> ASCII art -> local LLM (Ollama) -> ASCII art -> rendered image`

Runs fully offline against a local Ollama server. See `PLAN.md` for the plan, `LOG.md` for decisions.

## 0. Setup

In [ ]:
import time, json, base64, html, requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display

OLLAMA = "http://localhost:11434"
# sized for a 32 GB M1 Pro: every model <= ~9 GB, only one loaded at a time
MODELS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b", "llama3.1:8b",
          "qwen3:8b", "gemma2:9b", "gemma3:12b", "phi4:14b"]
VISION_MODELS = ["llava:7b", "gemma3:4b", "gemma3:12b"]   # pixel baseline
THINKING = ("qwen3",)                                    # turn reasoning off for these
CHARS  = " .:-=+*#%@"                                  # light -> dark
WIDTH  = 80                                            # ASCII columns

Path("images").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

installed = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]]
MODELS = [m for m in MODELS if m in installed]
print("installed:", installed)
VISION_MODELS = [m for m in VISION_MODELS if m in installed]
print("testing:  ", MODELS)
print("vision:   ", VISION_MODELS)

Sample images (only created if `images/` is empty), so the notebook runs without any downloads.

In [ ]:
if not any(Path("images").iterdir()):
    img = Image.new("L", (400, 400), 255); d = ImageDraw.Draw(img)
    d.ellipse((80, 80, 320, 320), fill=0)
    img.save("images/circle.png")

    img = Image.new("L", (400, 400), 255); d = ImageDraw.Draw(img)
    d.polygon([(50, 350), (200, 50), (350, 350)], fill=0)
    img.save("images/triangle.png")

    img = Image.new("L", (400, 400), 255); d = ImageDraw.Draw(img)
    d.ellipse((50, 50, 350, 350), fill=200, outline=0, width=8)
    d.ellipse((120, 130, 170, 180), fill=0); d.ellipse((230, 130, 280, 180), fill=0)
    d.arc((110, 160, 290, 300), 20, 160, fill=0, width=10)
    img.save("images/smiley.png")

    x = np.linspace(0, 255, 400).astype(np.uint8)
    Image.fromarray(np.tile(x, (400, 1))).save("images/gradient.png")
sorted(p.name for p in Path("images").iterdir())

## 1. Image -> ASCII

In [ ]:
def image_to_ascii(path, width=WIDTH):
    img = Image.open(path).convert("L")
    h = max(1, int(img.height / img.width * width * 0.5))   # chars are ~2x taller than wide
    px = np.array(img.resize((width, h))) / 255
    idx = ((1 - px) * (len(CHARS) - 1)).round().astype(int)
    return "\n".join("".join(CHARS[i] for i in row) for row in idx)

print(image_to_ascii("images/smiley.png"))

## 2. Call Ollama

In [ ]:
def ask(model, prompt, images=None):
    body = {"model": model, "prompt": prompt, "stream": False,
            "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 4096}}
    if images:
        body["images"] = images
    if model.startswith(THINKING):
        body["think"] = False
    t0 = time.time()
    r = requests.post(f"{OLLAMA}/api/generate", json=body, timeout=900).json()
    # a broken generation can come back with done=False and no stats: record it, don't crash
    ns = lambda k: r.get(k, 0) / 1e9
    return r.get("response", ""), {
        "done":       r.get("done", False),
        "total_s":    ns("total_duration") or time.time() - t0,
        "load_s":     ns("load_duration"),
        "in_tokens":  r.get("prompt_eval_count", 0),
        "out_tokens": r.get("eval_count", 0),
        "tok_per_s":  r.get("eval_count", 0) / ns("eval_duration") if r.get("eval_duration") else 0.0,
    }

def unload(model):   # free RAM before the next model loads
    requests.post(f"{OLLAMA}/api/generate", json={"model": model, "keep_alive": 0})

def warm_up(model):  # load once so load time doesn't skew results
    print(model, ask(model, "Say hi.")[1])

## 3. Tasks

In [ ]:
TASKS = {
    "describe": "Describe what this ASCII art shows in one sentence.",
    "flip":     "Mirror this ASCII art horizontally. Output only the ASCII art, same size.",
    "invert":   f"Invert brightness using this ramp (light to dark): '{CHARS}'. Output only the ASCII art, same size.",
    "edit":     "Add a small sun in the top-right corner. Output only the ASCII art, same size.",
}

def make_prompt(instruction, art):
    return f"{instruction}\n\n```\n{art}\n```"

## 4. Validate output

In [ ]:
def clean(text):
    lines = text.strip("\n").splitlines()
    return "\n".join(l for l in lines if not l.strip().startswith("```"))

def grid_ok(inp, out):
    a, b = inp.splitlines(), out.splitlines()
    return len(a) == len(b) and all(len(x) == len(y) for x, y in zip(a, b))

def expected(task, art):
    if task == "flip":
        return "\n".join(l[::-1] for l in art.splitlines())
    if task == "invert":
        return art.translate(str.maketrans(CHARS, CHARS[::-1]))
    return None

def char_accuracy(exp, out):
    a, b = exp.splitlines(), out.splitlines()
    hits = sum(x == y for la, lb in zip(a, b) for x, y in zip(la, lb))
    return hits / max(1, sum(map(len, a)))

## 5. ASCII -> image

In [ ]:
FONT = ImageFont.load_default()
CW, CH = 7, 12   # cell size; each char drawn in its own cell, so font needn't be monospace

def ascii_to_image(text, path=None):
    lines = text.splitlines() or [""]
    img = Image.new("L", (max(1, max(map(len, lines))) * CW, len(lines) * CH), 255)
    d = ImageDraw.Draw(img)
    for y, line in enumerate(lines):
        for x, c in enumerate(line):
            if c != " ":
                d.text((x * CW, y * CH), c, fill=0, font=FONT)
    if path:
        img.save(path)
    return img

ascii_to_image(image_to_ascii("images/smiley.png"))

## 6. Benchmark

In [ ]:
images = sorted(Path("images").glob("*.png"))
arts = {img: image_to_ascii(img) for img in images}
for img, art in arts.items():
    ascii_to_image(art, f"results/{img.stem}_input.png")

rows = []
for model in MODELS:
    warm_up(model)
    for img, art in arts.items():
        for task, instr in TASKS.items():
            out, stats = ask(model, make_prompt(instr, art))
            out = out.strip() if task == "describe" else clean(out)
            row = {"image": img.name, "model": model, "task": task, **stats, "output": out}
            if task != "describe":
                row["grid_ok"] = grid_ok(art, out)
                exp = expected(task, art)
                row["char_acc"] = char_accuracy(exp, out) if exp else None
                ascii_to_image(out, f"results/{img.stem}_{model.replace(':', '-')}_{task}.png")
            rows.append(row)
    unload(model)
    print(f"done: {model}")

df = pd.DataFrame(rows)
df.to_csv("results/benchmarks.csv", index=False)

## 7. Results

Per image: the input on the left, then one row per model with every task response shown in full. Stats are the small grey line under each response.

In [ ]:
def img_tag(path, width=160):
    b64 = base64.b64encode(Path(path).read_bytes()).decode()
    return f'<img src="data:image/png;base64,{b64}" width="{width}">'

def stat_line(r):
    bits = [f"{r['total_s']:.1f}s", f"{r['tok_per_s']:.0f} tok/s", f"{r['out_tokens']} tok"]
    if r["task"] != "describe":
        bits.append("grid ✓" if r["grid_ok"] else "grid ✗")
    if pd.notna(r.get("char_acc")):
        bits.append(f"acc {r['char_acc']:.2f}")
    if not r["done"]:
        bits.append("<b>FAILED</b>")
    return " · ".join(bits)

CSS = """<style>
.poc {border-collapse: collapse; margin-bottom: 2em}
.poc td, .poc th {border: 1px solid #ccc; padding: 6px; vertical-align: top; text-align: left}
.poc pre {font-size: 7px; line-height: 1; margin: 0; white-space: pre; max-width: 420px; overflow-x: auto; max-height: none}
.poc .txt {font-size: 13px; white-space: normal; min-width: 200px; max-width: 260px}
.poc .stat {color: #888; font-size: 11px; margin-top: 4px}
</style>"""

def report(df):
    out = [CSS]
    for img, art in arts.items():
        sub = df[df.image == img.name]
        models = list(dict.fromkeys(sub.model))
        t = [f"<h3>{img.name}</h3><div style='overflow-x:auto'><table class='poc'>",
             "<tr><th>input</th><th>model</th>" + "".join(f"<th>{k}</th>" for k in TASKS) + "</tr>"]
        for i, m in enumerate(models):
            cells = ""
            if i == 0:
                cells += (f"<td rowspan='{len(models)}'>{img_tag(img)}<br><br>"
                          f"<pre>{html.escape(art)}</pre></td>")
            cells += f"<td><b>{m}</b></td>"
            for k in TASKS:
                r = sub[(sub.model == m) & (sub.task == k)].iloc[0]
                body = (f"<div class='txt'>{html.escape(r.output)}</div>" if k == "describe"
                        else f"<pre>{html.escape(r.output)}</pre>")
                cells += f"<td>{body}<div class='stat'>{stat_line(r)}</div></td>"
            t.append(f"<tr>{cells}</tr>")
        out.append("".join(t) + "</table></div>")
    page = "".join(out)
    Path("results/report.html").write_text(page)
    display(HTML(page))

report(df)

### Stats

In [ ]:
df.groupby(["model", "task"])[["done", "total_s", "in_tokens", "out_tokens", "tok_per_s", "grid_ok", "char_acc"]].mean()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df.groupby("model")["tok_per_s"].mean().plot.bar(ax=ax[0], title="tokens / sec")
df[df.task != "describe"].groupby("model")["grid_ok"].mean().plot.bar(ax=ax[1], title="grid shape kept", ylim=(0, 1))
plt.tight_layout(); plt.show()

### Baseline: vision models on the real image\n\nCompares tokens/time/answers for pixels vs. ASCII on the `describe` task. Skipped if no `VISION_MODELS` are installed.

In [ ]:
base = []
for vm in VISION_MODELS:
    warm_up(vm)
    for img, art in arts.items():
        b64 = base64.b64encode(img.read_bytes()).decode()
        out, s = ask(vm, "Describe this image in one sentence.", images=[b64])
        base.append({"model": vm, "image": img.name, "input": "pixels", **s, "answer": out.strip()})
        out, s = ask(vm, make_prompt(TASKS["describe"], art))
        base.append({"model": vm, "image": img.name, "input": "ascii", **s, "answer": out.strip()})
    unload(vm)

if base:
    bdf = pd.DataFrame(base)
    bdf.to_csv("results/vision_baseline.csv", index=False)
    display(bdf.groupby(["model", "input"])[["total_s", "in_tokens", "out_tokens"]].mean())
    with pd.option_context("display.max_colwidth", None):   # show full answers
        display(bdf[["model", "image", "input", "answer"]])
else:
    print("no vision models installed; skipping baseline")